# Q-learning
Here's some pseudocode from Stanford's CS234 with Emma Brunskill (which I took a few years ago and subsequently forgot all of, sorry Emma):

<img src="Q_learning_pseudocode.png" width="700" />

Essentially, we want to keep a table of Q values, Q(s,a), where each Q value represents *the expected total (discounted) future reward you'll get if you take action a in state s, and then act optimally forever after*. To put it simply: Q value big = good, Q value small = bad. Q table = lookup table of Q value for each (state, action) pair.
I used the word *table* on purpose: in order to create our Q table, we have to discretize our continuous state space into discrete boxes.
I'll be using the same "box" representation as Sutton & Barto did in [Neuronlike adaptive elements that can solve difficult learning control problems](https://ieeexplore.ieee.org/document/6313077) (which I reimplemented [here](../ase_ace/ASE_ACE.ipynb)), which itself is taken from Michie & Chambers'1968 paper [Boxes: An Experiment in Adaptive Control](https://aitopics.org/download/aiclassics:64BFF71C). 
This discretizes the cartpole state space into 162 boxes. Of course, with modern computing resources we could afford to discretize into plenty more than that, but let's start simple.

In [43]:
import numpy as np
import jax
import jax.numpy as jnp

N_BOXES = 162

def get_box(observation):
    """
    takes in state (x, x_dot, theta, theta_dot) and discretizes into 162 bins as specified in the paper, 
    from Michie & Chambers' boxes system
    returns box index ("state" index)
    """
    x, x_dot, theta, theta_dot = observation

    # (3) x: +/- 0.8, +/- 2.4
    if x < -0.8:
        x_bin = 0
    elif x < 0.8:
        x_bin = 1
    else:
        x_bin = 2
    
    # (6) theta: 0, +/- 1, +/- 6, +/- 12
    theta_deg = jnp.degrees(theta)  # paper specifies thresholds in degrees
    if theta_deg < -6:
        theta_bin = 0
    elif theta_deg < -1:
        theta_bin = 1
    elif theta_deg < 0:
        theta_bin = 2
    elif theta_deg < 1:
        theta_bin = 3
    elif theta_deg < 6:
        theta_bin = 4
    else:
        theta_bin = 5
    
    # (3) x_dot: +/- 0.5, +/- inf
    if x_dot < -0.5:
        x_dot_bin = 0
    elif x_dot < 0.5:
        x_dot_bin = 1
    else:
        x_dot_bin = 2

    # (3) theta_dot: +/0 50, +/- inf
    theta_dot_deg = jnp.degrees(theta_dot)
    if theta_dot_deg < -50:
        theta_dot_bin = 0
    elif theta_dot_deg < 50:
        theta_dot_bin = 1
    else:
        theta_dot_bin = 2

    # convert into idx of 3 * 6 * 3 * 3 = 162 bins
    box_idx = (
        x_bin * (3 * 6 * 3)
        + x_dot_bin * (6 * 3)
        + theta_bin * 3
        + theta_dot_bin
    )
    return box_idx

In [44]:
from typing import NamedTuple
import jax
import jax.numpy as jnp

N_ACTIONS = 2
EPSILON = 0.1
ALPHA = 0.3
GAMMA = 0.99

class QLearnerState(NamedTuple):
    q_table: jnp.ndarray  # (n_boxes, n_actions)
    eps: float
    alpha: float
    gamma: float

def init_q_learner(n_states=N_BOXES, n_actions=N_ACTIONS, eps=EPSILON, alpha=ALPHA, gamma=GAMMA):
    return QLearnerState(
        q_table=jnp.ones((n_states, n_actions)) * 100, # NOTE: initializing the q table with large values drives early exploration!
        eps=eps, alpha=alpha, gamma=gamma,
    )

def get_e_greedy_action(state, observation, act_key):
    explore_key, choice_key = jax.random.split(act_key)
    if jax.random.uniform(explore_key) < state.eps:
        # explore! fancy way of saying "pick 0 or 1 at random"
        return int(jax.random.randint(choice_key, (), 0, N_ACTIONS))
    # exploit! take the action that, based on this observation, we currently think will give us Best Future Returns
    box = get_box(observation)
    return int(jnp.argmax(state.q_table[box]))

def update_q_table(state, prev_observation, action, observation, reward, terminated, truncated, info):
    """
    update according to line 6 of the pseudocode from the image above;
    Q(s, a) <- Q(s, a) + alpha * (target - Q(s, a)) where s is the previous state and a is the action we just took
    except if the episode ends, our target is just the reward instead of expected discounted reward under optimal action
    """
    prev_box = get_box(prev_observation)
    new_box = get_box(observation)
    done = terminated or truncated
    target = reward if done else reward + state.gamma * jnp.max(state.q_table[new_box])
    update = state.alpha * (target - state.q_table[prev_box, action])
    new_q_table = state.q_table.at[prev_box, action].add(update)
    return state._replace(q_table=new_q_table)

In [45]:

no_op = lambda *args, **kwargs: None
def pass_thru(state, *args, **kwargs): return state

# state = init_agent()
init_agent = init_q_learner

# state = reset_agent(state)
reset_agent = pass_thru

# action = get_action(state, observation, act_key)
get_action = get_e_greedy_action

# state = update(state, observation, reward, terminated, truncated, info)
update = update_q_table

model_name = "Boxes Q-learning"

In [46]:
import gymnasium as gym
import jax
import jax.numpy as jnp
import numpy as np
from tqdm import tqdm

env = gym.make(
    "CartPole-v1",
    render_mode="none",
)

N_TRIALS = 100
N_REPS = 10

# prng key
key = jax.random.PRNGKey(0)

all_trial_results = []

for rep in range(N_REPS):
    # independent key per rep, derived from rep index via fold_in
    key = jax.random.fold_in(jax.random.PRNGKey(0), rep)

    # TODO
    state = init_agent()

    trial_results = []

    pbar = tqdm(range(N_TRIALS), desc=f"repeat {rep}")
    for trial_idx in pbar:
        observation, info = env.reset()

        # TODO optionally reset the agent
        # NOTE added state as a param here so reset agent can also be pass thru
        state = reset_agent(state)

        episode_over = False
        steps = 0

        while not episode_over:
            # this is the nice jax way to get a new random seed which is properly independent
            key, act_key = jax.random.split(key)

            # TODO get action (use act_key if it is random)
            action = get_action(state, observation, act_key)

            prev_observation = observation

            # execute action in env
            observation, reward, terminated, truncated, info = env.step(action)
            steps += 1

            # TODO do something with the new info
            # NOTE updated this fn to take in prev_observation and action
            state = update(state, prev_observation, action, observation, reward, terminated, truncated, info)

            episode_over = terminated or truncated

        trial_results.append(steps)
        pbar.set_postfix({"steps": steps})
    
    pbar.close()
    all_trial_results.append(trial_results)

env.close()

all_trial_results = np.array(all_trial_results)  # shape (N_REPS, N_TRIALS)
np.save(f"{model_name}_100_trial_results.npy", all_trial_results)

# Compute average + std steps alive over the last K trials across reps
K = 5
final_trial_steps = all_trial_results[:, -K]  # shape (N_REPS,K)
mean_final = final_trial_steps.mean()
std_final = final_trial_steps.std()
print(f"Final trial (#{N_TRIALS}) steps alive across {N_REPS} reps: "
      f"{mean_final:.2f} ± {std_final:.2f}")

repeat 9: 100%|██████████| 100/100 [00:08<00:00, 12.40it/s, steps=106]

Final trial (#100) steps alive across 10 reps: 131.90 ± 66.21


In [48]:
import plotly.graph_objects as go
import numpy as np

def plot_trial_results(model_name, mean_curve, std_curve=None):
    """Plot mean steps-until-failure per trial, with optional std band."""
    trials = np.arange(len(mean_curve))
    fig = go.Figure()
    title=f"{model_name} Learning Curve"

    if std_curve is not None:
        # Upper bound (invisible line, anchors the fill)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve + std_curve,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ))
        # Lower bound (fills down to upper bound)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve - std_curve,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(0, 100, 200, 0.2)",
            name="±1 std",
            hoverinfo="skip",
        ))

    # Mean line on top
    fig.add_trace(go.Scatter(
        x=trials,
        y=mean_curve,
        mode="lines+markers",
        name="Mean steps until failure",
        line=dict(color="rgb(0, 100, 200)"),
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Trial",
        yaxis_title="Steps until failure",
        template="plotly_white",
    )
    config = {
    'toImageButtonOptions': {
        'format': 'png', # or 'svg', 'jpeg', 'webp'
        'scale': 3 # triple the resolution of the saved img
    }
    }

    fig.show(config=config)

mean_curve = all_trial_results.mean(axis=0)
std_curve = all_trial_results.std(axis=0)
plot_trial_results(model_name, mean_curve, std_curve)